In [ ]:
# Colab bootstrap: install requirements and stage the course data next to the notebook
import shutil
import subprocess
import sys
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    REPO_URL = "https://github.com/griu/ai-ml-finance-hands-on"
    REPO_DIR = Path("/content/ai-ml-finance-hands-on")
    NOTEBOOK_DIR = Path.cwd()
    DATA_TARGET = NOTEBOOK_DIR / "data"
    REQUIREMENTS_FILE = REPO_DIR / "requirements-colab.txt"

    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(REQUIREMENTS_FILE)],
        check=True,
    )

    shutil.copytree(REPO_DIR / "data", DATA_TARGET, dirs_exist_ok=True)
    print(f"Colab environment detected. Data copied to: {DATA_TARGET.resolve()}")
else:
    print("Local environment detected. Colab bootstrap skipped.")


# NB05c — Compact Supervised Methods: Naïve Bayes, SVM and k-NN

**Role:** Compact-extension  
**Manual section:** 2.3.1.c — Naïve Bayes, SVM and k-NN  
**Prerequisites:** NB05

---

## Purpose of this notebook

This compact-extension notebook broadens the supervised ML map with three additional methods beyond the logistic regression baseline (NB05) and tree-based models (NB05b). Each illustrates a different learning logic: probabilistic independence (Naïve Bayes), maximum-margin separation (SVM), and instance-based reasoning (k-NN).

These methods are presented as benchmarks and niche alternatives rather than as the main production candidates for tabular finance problems.

**Dataset:** `dataset_C_attrition_model_ready.csv` (the shared modelling table from NB04)

**What you will learn:**
- How Gaussian Naïve Bayes, SVM and k-NN work at an intuitive level
- Why standardisation matters for distance-based and margin-based methods
- How these models compare with the baselines from NB05 and NB05b

## Section 0 — Why these models matter

These methods illustrate different learning logics: **Naïve Bayes** uses a probabilistic independence assumption, **SVM** searches for a strong separating margin, and **k-NN** predicts by looking at nearby cases. In applied finance, they are often benchmark or niche models rather than the final production choice.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

from pathlib import Path
_candidates = [Path('data/processed'), Path('../data/processed')]
DATA_DIR = next((p for p in _candidates if p.is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Cannot locate data/processed/. Launch from project root or notebooks/.')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score


In [ ]:
# ── Load the shared modelling table built in NB04 ──────────────────────
df = pd.read_csv(DATA_DIR / 'dataset_C_attrition_model_ready.csv')

# Use the same features and target as NB05
target = 'attrition_flag'
drop_cols = [c for c in ['customer_id', target] if c in df.columns]
X = df.drop(columns=drop_cols)
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f"Modelling table loaded: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}")
X_train.shape, X_test.shape

In [ ]:
def evaluate(name, y_true, y_pred, y_prob):
    return {'model': name, 'accuracy': accuracy_score(y_true, y_pred), 'precision': precision_score(y_true, y_pred, zero_division=0), 'recall': recall_score(y_true, y_pred, zero_division=0), 'f1': f1_score(y_true, y_pred, zero_division=0), 'auc': roc_auc_score(y_true, y_prob)}

## Section 1 — Gaussian Naïve Bayes

### How Naïve Bayes works

Gaussian Naïve Bayes assumes that each feature contributes independently to the probability of the target class. For each class, it estimates the mean and variance of every feature so that it can compute the likelihood of a new observation belonging to that class.

**Why "naïve"?** The independence assumption is almost never true in practice — features like balance and number of products are likely correlated. Yet the model often works surprisingly well as a fast, simple baseline, especially when data is limited.

**Finance intuition:** think of it as asking "given what we know about each variable separately, how likely is this customer to leave?" It ignores interactions between variables but sometimes that simplicity is enough.

In [ ]:
from sklearn.naive_bayes import GaussianNB
gnb = GaussianNB()
gnb.fit(X_train_sc, y_train)
prob_nb = gnb.predict_proba(X_test_sc)[:, 1]
pred_nb = gnb.predict(X_test_sc)
res_nb = evaluate('GaussianNB', y_test, pred_nb, prob_nb)
res_nb

## Section 2 — Support Vector Machine

### How SVM works

A Support Vector Machine looks for the **decision boundary** (hyperplane) that maximises the margin between the two classes. The "support vectors" are the observations closest to this boundary — they define its position.

With an **RBF kernel**, SVM can learn non-linear boundaries by implicitly mapping the data into a higher-dimensional space. Standardisation is essential because SVM is sensitive to the scale of features.

**Finance intuition:** SVM asks "what is the widest corridor I can draw between customers who leave and customers who stay?" It tends to be effective when classes are clearly separable, but less interpretable than logistic regression or trees.

In [ ]:
from sklearn.svm import SVC
svm = SVC(kernel='rbf', C=1.0, probability=True, random_state=42)
svm.fit(X_train_sc, y_train)
prob_svm = svm.predict_proba(X_test_sc)[:, 1]
pred_svm = svm.predict(X_test_sc)
res_svm = evaluate('SVM (rbf)', y_test, pred_svm, prob_svm)
res_svm

## Section 3 — k-Nearest Neighbours

### How k-NN works

k-Nearest Neighbours makes no assumptions about the shape of the decision boundary. For each new observation, it finds the **k closest cases** in the training data and assigns the majority class among those neighbours.

**Why standardisation matters:** if `balance` ranges from 0 to 200,000 and `num_products` ranges from 1 to 4, distance calculations would be dominated by balance unless we standardise.

**Finance intuition:** k-NN asks "which existing customers does this person look most like, and what happened to them?" It is intuitive but can be slow on large datasets and sensitive to noisy features.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(n_neighbors=25)
knn.fit(X_train_sc, y_train)
prob_knn = knn.predict_proba(X_test_sc)[:, 1]
pred_knn = knn.predict(X_test_sc)
res_knn = evaluate('k-NN (k=25)', y_test, pred_knn, prob_knn)
res_knn

## Section 4 — Compare the results

In [ ]:
comparison = pd.DataFrame([res_nb, res_svm, res_knn]).round(3).sort_values('auc', ascending=False)
comparison

In [ ]:
comparison.set_index('model')[['auc', 'recall', 'f1']].plot(kind='bar')
plt.title('Compact comparison: Naïve Bayes, SVM, k-NN')
plt.xticks(rotation=20)
plt.ylim(0, 1)
plt.show()

## What you have learned

- Gaussian Naïve Bayes provides a fast probabilistic baseline using an independence assumption.
- SVM searches for maximum-margin separation and can model non-linear boundaries with kernels.
- k-NN predicts by similarity to training cases, making it intuitive but scale-sensitive.
- All three methods use the same shared modelling table and train/test split as NB05 and NB05b, making metric comparison straightforward.

### How do these compare with the core models?

The logistic regression baseline (NB05) and tree-based ensembles (NB05b) remain the primary methods for tabular finance problems in this course. The three methods in this notebook broaden the map but are rarely the first choice for production credit or attrition models.

---

**Project chain:** NB04 (build table) → NB05 (baseline) → NB05b (trees) → **NB05c (compact methods)** → NB06 (validate & interpret)  
**Current position:** NB05c

*This is a compact-extension notebook supporting manual section 2.3.1.c.*